# 05 — Tích hợp Hybrid Classification, Tối ưu Ngưỡng & Gán Nhãn Email Mới (Demo)

**Phụ trách:** Trưởng nhóm (Thành viên 5) — Nhánh `lead/hybrid-integration`

## Mục tiêu Notebook
1. Tích hợp điểm thành phần từ **Rule Engine** (`S_rule`) và **Statistics Engine** (`S_stat`) thành mô hình lai **Hybrid (`S_total = S_rule + S_stat`)**.
2. Tối ưu 3 ngưỡng phân loại (`T_rule`, `T_stat`, `T_combined`) **chỉ trên tập `data/validation.csv`** (chống Data Leakage) và lưu vào `config/settings.json`.
3. Áp ngưỡng cố định lên tập **`data/test.csv`** độc lập và xuất `outputs/hybrid_test_predictions.csv` cho Thành viên 4 đánh giá.
4. Chạy **gán nhãn tự động** cho 30 email mới chưa có nhãn trong `data/unlabeled_sample.csv` → xuất `outputs/labeled_new_emails.csv`.
5. Cung cấp hàm `predict_single_email(subject, body, mode="combined")` để demo trực tiếp khi thuyết trình.

> **Lưu ý học thuật:** Các điểm số `S_rule`, `S_stat`, `S_total` là **điểm phân loại có ngưỡng**, KHÔNG phải xác suất `% Spam`.

In [1]:
import csv
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classifier import classify_email, load_settings, save_settings
from src.evaluation import check_data_leakage, evaluate, evaluate_majority_baseline
from src.models import Email
from src.preprocessing import load_emails_from_csv, prepare_email_tokens
from src.rules import load_rules, score_rules
from src.statistics import load_statistics, score_statistics, train_statistics

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
for rel in ["data/train.csv", "data/validation.csv", "data/test.csv", "data/unlabeled_sample.csv"]:
    assert (PROJECT_ROOT / rel).is_file(), f"Thiếu tệp {rel}. Hãy chạy: python scripts/setup_data.py"
print("Đã xác nhận đầy đủ các tệp dữ liệu chuẩn!")

PROJECT_ROOT: D:\uth\Spam_Email
Đã xác nhận đầy đủ các tệp dữ liệu chuẩn!


## 1. Tải dữ liệu & Nạp nguồn điểm Rule Engine + Statistics Engine

- Nếu các nhánh của **Nguyễn Kiệt** (`config/rules.json`, `outputs/rule_*_scores.csv`) và **leehuy** (`data/processed/keyword_stats.json`, `outputs/stat_*_scores.csv`) đã merge về `main`, notebook tự động đọc trực tiếp file chính thức.
- Nếu chưa merge, notebook tự động chạy chế độ dự phòng bằng bộ hàm chuẩn trong `src/` (không ghi đè file của thành viên khác).

In [2]:
FALLBACK_RULES = [
    {"id": "R001", "pattern": "claim your prize", "weight": 4.0, "description": "Cụm mời nhận thưởng"},
    {"id": "R002", "pattern": "limited offer", "weight": 2.5, "description": "Quảng cáo giới hạn"},
    {"id": "R003", "pattern": "click here", "weight": 2.5, "description": "Kêu gọi bấm liên kết"},
    {"id": "R004", "pattern": "act now", "weight": 2.5, "description": "Thúc giục hành động gấp"},
    {"id": "R005", "pattern": "winner", "weight": 3.0, "description": "Từ khóa trúng thưởng"},
    {"id": "R006", "pattern": "free", "weight": 1.5, "description": "Quảng cáo miễn phí"},
    {"id": "R007", "pattern": "viagra", "weight": 4.0, "description": "Spam dược phẩm"},
    {"id": "R008", "pattern": "pills", "weight": 3.0, "description": "Spam thuốc"},
    {"id": "R009", "pattern": "pharmacy", "weight": 3.0, "description": "Spam nhà thuốc online"},
    {"id": "R010", "pattern": "cheap", "weight": 2.0, "description": "Quảng cáo giá rẻ"},
    {"id": "R011", "pattern": "prescription", "weight": 2.5, "description": "Spam thuốc kê đơn"},
    {"id": "R012", "pattern": "guaranteed", "weight": 2.0, "description": "Cam kết quá mức"},
    {"id": "R013", "pattern": "cash", "weight": 2.0, "description": "Tiền mặt / thưởng tiền"},
    {"id": "R014", "pattern": "money", "weight": 1.5, "description": "Kiếm tiền nhanh"},
    {"id": "R015", "pattern": "unsubscribe", "weight": 1.5, "description": "Email quảng cáo hàng loạt"},
    {"id": "R016", "pattern": "investment", "weight": 1.5, "description": "Mời chào đầu tư"},
    {"id": "R017", "pattern": "enron", "weight": -2.5, "description": "Email nội bộ Enron"},
    {"id": "R018", "pattern": "houston", "weight": -1.5, "description": "Địa điểm văn phòng nội bộ"},
    {"id": "R019", "pattern": "meeting agenda", "weight": -2.5, "description": "Lịch họp công việc"},
    {"id": "R020", "pattern": "attached file", "weight": -2.0, "description": "Tài liệu đính kèm công việc"},
    {"id": "R021", "pattern": "meter", "weight": -2.0, "description": "Thuật ngữ vận hành khí đốt Enron"},
    {"id": "R022", "pattern": "hpl", "weight": -2.5, "description": "Đơn vị nghiệp vụ HPL"},
    {"id": "R023", "pattern": "ect", "weight": -2.5, "description": "Định danh nội bộ ECT"},
    {"id": "R024", "pattern": "forwarded", "weight": -1.5, "description": "Chuyển tiếp thư công việc"},
]

train_emails, _ = load_emails_from_csv(str(PROJECT_ROOT / "data/train.csv"), require_label=True)
val_emails, _ = load_emails_from_csv(str(PROJECT_ROOT / "data/validation.csv"), require_label=True)
test_emails, _ = load_emails_from_csv(str(PROJECT_ROOT / "data/test.csv"), require_label=True)
unlabeled_emails, _ = load_emails_from_csv(str(PROJECT_ROOT / "data/unlabeled_sample.csv"), require_label=False)

# Kiểm tra chống Data Leakage
leak_report = check_data_leakage(train_emails, val_emails, test_emails)
assert not leak_report["has_leakage"], f"Phát hiện Data Leakage: {leak_report}"
print(f"Số lượng email: train={len(train_emails)}, val={len(val_emails)}, test={len(test_emails)}, unlabeled={len(unlabeled_emails)} | Leakage: False")

# Nạp bộ quy tắc (ưu tiên config/rules.json của Nguyễn Kiệt nếu đã có)
rules_path = PROJECT_ROOT / "config/rules.json"
if rules_path.is_file():
    rules = load_rules(str(rules_path))
    print(f"Đã nạp {len(rules)} quy tắc chính thức từ config/rules.json")
else:
    rules = FALLBACK_RULES
    print(f"[Dự phòng] Chưa có config/rules.json từ nhánh Kiệt — đang dùng {len(rules)} quy tắc mẫu.")

# Nạp bảng thống kê (ưu tiên data/processed/keyword_stats.json của leehuy nếu đã có)
stats_path = PROJECT_ROOT / "data/processed/keyword_stats.json"
if stats_path.is_file():
    stats_model = load_statistics(str(stats_path))
    print(f"Đã nạp bảng thống kê chính thức từ data/processed/keyword_stats.json (vocab={stats_model.get('vocab_size')})")
else:
    stats_model = train_statistics(train_emails, min_df=3)
    print(f"[Dự phòng] Chưa có keyword_stats.json từ nhánh leehuy — tự huấn luyện trên train.csv (vocab={stats_model['vocab_size']}).")

Số lượng email: train=350, val=75, test=75, unlabeled=30 | Leakage: False
[Dự phòng] Chưa có config/rules.json từ nhánh Kiệt — đang dùng 24 quy tắc mẫu.
[Dự phòng] Chưa có keyword_stats.json từ nhánh leehuy — tự huấn luyện trên train.csv (vocab=1876).


In [3]:
def _load_score_csv(path: Path, score_col: str, ev_col: str) -> dict[str, tuple[float, str]]:
    mapping = {}
    with path.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            mapping[row["email_id"]] = (float(row[score_col]), row.get(ev_col, ""))
    return mapping


def build_combined_records(emails: list[Email], split_name: str) -> list[dict]:
    rule_csv = PROJECT_ROOT / f"outputs/rule_{split_name}_scores.csv"
    stat_csv = PROJECT_ROOT / f"outputs/stat_{split_name}_scores.csv"

    rule_map = _load_score_csv(rule_csv, "rule_score", "matched_rules") if rule_csv.is_file() else None
    stat_map = _load_score_csv(stat_csv, "stat_score", "top_keywords") if stat_csv.is_file() else None

    records = []
    for em in emails:
        rule_tokens, stat_tokens = prepare_email_tokens(em)

        if rule_map is not None and em.email_id in rule_map:
            r_score, r_ev = rule_map[em.email_id]
        else:
            r_score, r_matches = score_rules(rule_tokens, rules)
            r_ev = "; ".join(f"{m.rule_id}:{m.pattern}({m.weight:+.1f})" for m in r_matches)

        if stat_map is not None and em.email_id in stat_map:
            s_score, s_ev = stat_map[em.email_id]
        else:
            s_score, s_matches = score_statistics(stat_tokens, stats_model, top_k=5)
            s_ev = "; ".join(f"{k.token}({k.weight:+.2f})" for k in s_matches)

        records.append({
            "email_id": em.email_id,
            "label": em.label,
            "label_num": 1 if em.label == "spam" else 0,
            "rule_score": round(r_score, 4),
            "stat_score": round(s_score, 4),
            "combined_score": round(r_score + s_score, 4),
            "matched_rules": r_ev,
            "top_keywords": s_ev,
        })
    return records

val_records = build_combined_records(val_emails, "val")
test_records = build_combined_records(test_emails, "test")
print(f"Đã tổng hợp điểm: validation={len(val_records)} dòng, test={len(test_records)} dòng.")

Đã tổng hợp điểm: validation=75 dòng, test=75 dòng.


## 2. Tối ưu Ngưỡng phân loại trên Validation Set (`data/validation.csv`)

Quét lưới ngưỡng từ `-3.0` đến `+4.0` (bước `0.25`) cho cả 3 chế độ (`rule_only`, `statistics_only`, `combined`), chọn ngưỡng đạt **F1-score cao nhất** trên `validation.csv` (ưu tiên phụ Accuracy), sau đó cố định vào `config/settings.json`.

In [4]:
candidate_thresholds = [round(-3.0 + 0.25 * i, 2) for i in range(29)]  # -3.00 -> +4.00
y_val_true = [r["label"] for r in val_records]

mode_to_col = {
    "rule_only": "rule_score",
    "statistics_only": "stat_score",
    "combined": "combined_score",
}

best_thresholds = {}
tuning_history = {}

for mode, col in mode_to_col.items():
    history = []
    best_item = None
    for th in candidate_thresholds:
        y_pred = ["spam" if r[col] >= th else "ham" for r in val_records]
        m = evaluate(y_val_true, y_pred)
        item = {"threshold": th, "f1": m["f1"], "accuracy": m["accuracy"], "precision": m["precision"], "recall": m["recall"], "fpr": m["fpr"]}
        history.append(item)
        key = (item["f1"], item["accuracy"], -abs(th - 1.0))
        if best_item is None or key > best_item[0]:
            best_item = (key, item)
    tuning_history[mode] = history
    best_thresholds[mode] = best_item[1]["threshold"]
    b = best_item[1]
    print(f"[{mode:15s}] Best T = {b['threshold']:+.2f} | Val F1={b['f1']:.4f} | Acc={b['accuracy']:.4f} | Prec={b['precision']:.4f} | Rec={b['recall']:.4f} | FPR={b['fpr']:.4f}")

# Lưu ngưỡng cố định vào config/settings.json
settings = load_settings(str(PROJECT_ROOT / "config/settings.json"))
settings["thresholds"] = best_thresholds
save_settings(settings, str(PROJECT_ROOT / "config/settings.json"))
print("Đã lưu 3 ngưỡng tối ưu vào config/settings.json:", best_thresholds)

# Vẽ biểu đồ F1 theo ngưỡng (nếu đã cài matplotlib)
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(9, 5))
    for mode, hist in tuning_history.items():
        xs = [h["threshold"] for h in hist]
        ys = [h["f1"] for h in hist]
        plt.plot(xs, ys, marker="o", markersize=3, label=f"{mode} (T*={best_thresholds[mode]:+.2f})")
    plt.title("Tối ưu ngưỡng phân loại theo F1-score trên Validation Set")
    plt.xlabel("Ngưỡng phân loại (Threshold)")
    plt.ylabel("F1-score")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    fig_path = PROJECT_ROOT / "outputs/figures/hybrid_threshold_tuning.png"
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(fig_path, dpi=150)
    plt.close()
    print(f"Đã lưu biểu đồ tối ưu ngưỡng: {fig_path}")
except ImportError:
    print("[Lưu ý] Chưa cài matplotlib — chạy 'pip install -r requirements.txt' để xuất biểu đồ PNG.")

[rule_only      ] Best T = +0.00 | Val F1=0.8636 | Acc=0.8400 | Prec=0.7600 | Rec=1.0000 | FPR=0.3243
[statistics_only] Best T = -0.75 | Val F1=0.9474 | Acc=0.9467 | Prec=0.9474 | Rec=0.9474 | FPR=0.0541
[combined       ] Best T = -0.75 | Val F1=0.9487 | Acc=0.9467 | Prec=0.9250 | Rec=0.9737 | FPR=0.0811
Đã lưu 3 ngưỡng tối ưu vào config/settings.json: {'rule_only': 0.0, 'statistics_only': -0.75, 'combined': -0.75}
[Lưu ý] Chưa cài matplotlib — chạy 'pip install -r requirements.txt' để xuất biểu đồ PNG.


## 3. Áp ngưỡng cố định lên Tập Validation & Tập Test Độc Lập (`data/test.csv`)

Xuất `outputs/hybrid_val_predictions.csv` và `outputs/hybrid_test_predictions.csv` đúng **11 cột chuẩn** để **Phí Đặng Nguyên Huy...** chạy Notebook 04 đánh giá chi tiết, vẽ Confusion Matrix và phân tích lỗi.

In [5]:
PRED_COLS = [
    "email_id", "label", "label_num",
    "rule_score", "stat_score", "combined_score",
    "pred_rule", "pred_stat", "pred_combined",
    "matched_rules", "top_keywords",
]

def apply_thresholds_and_export(records: list[dict], thresholds: dict, out_path: Path) -> list[dict]:
    enriched = []
    for r in records:
        row = dict(r)
        row["pred_rule"] = "spam" if r["rule_score"] >= thresholds["rule_only"] else "ham"
        row["pred_stat"] = "spam" if r["stat_score"] >= thresholds["statistics_only"] else "ham"
        row["pred_combined"] = "spam" if r["combined_score"] >= thresholds["combined"] else "ham"
        enriched.append(row)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=PRED_COLS)
        writer.writeheader()
        writer.writerows(enriched)
    return enriched

val_preds = apply_thresholds_and_export(
    val_records, best_thresholds, PROJECT_ROOT / "outputs/hybrid_val_predictions.csv"
)
test_preds = apply_thresholds_and_export(
    test_records, best_thresholds, PROJECT_ROOT / "outputs/hybrid_test_predictions.csv"
)

# Tóm tắt kết quả thực nghiệm trên tập Test độc lập
y_test_true = [r["label"] for r in test_preds]
train_labels = [e.label for e in train_emails]
baseline = evaluate_majority_baseline(train_labels, y_test_true)
print(f"\n=== KẾT QUẢ THỰC NGHIỆM TRÊN TẬP TEST ({len(test_preds)} EMAIL) ===")
print(f"{'Baseline (' + baseline['baseline_label'] + ')':18s} | Acc={baseline['accuracy']:.4f} | Prec={baseline['precision']:.4f} | Rec={baseline['recall']:.4f} | F1={baseline['f1']:.4f} | FPR={baseline['fpr']:.4f}")
for label_name, col in [("Rule-only", "pred_rule"), ("Statistics-only", "pred_stat"), ("Combined (Hybrid)", "pred_combined")]:
    m = evaluate(y_test_true, [r[col] for r in test_preds])
    print(f"{label_name:18s} | Acc={m['accuracy']:.4f} | Prec={m['precision']:.4f} | Rec={m['recall']:.4f} | F1={m['f1']:.4f} | FPR={m['fpr']:.4f} | TP={m['tp']} TN={m['tn']} FP={m['fp']} FN={m['fn']}")


=== KẾT QUẢ THỰC NGHIỆM TRÊN TẬP TEST (75 EMAIL) ===
Baseline (ham)     | Acc=0.5067 | Prec=0.0000 | Rec=0.0000 | F1=0.0000 | FPR=0.0000
Rule-only          | Acc=0.8933 | Prec=0.8222 | Rec=1.0000 | F1=0.9024 | FPR=0.2105 | TP=37 TN=30 FP=8 FN=0
Statistics-only    | Acc=0.9067 | Prec=1.0000 | Rec=0.8108 | F1=0.8955 | FPR=0.0000 | TP=30 TN=38 FP=0 FN=7
Combined (Hybrid)  | Acc=0.9333 | Prec=0.9706 | Rec=0.8919 | F1=0.9296 | FPR=0.0263 | TP=33 TN=37 FP=1 FN=4


## 4. Gán nhãn tự động cho 30 Email Mới chưa có nhãn (`data/unlabeled_sample.csv`)

Sử dụng `classify_email(..., mode="combined")` từ `src/classifier.py` để tự động gán nhãn `predicted_label` (`spam`/`ham`), `predicted_label_num` (`1`/`0`) kèm bằng chứng giải thích và lưu vào `outputs/labeled_new_emails.csv`.

In [6]:
labeled_rows = []
for em in unlabeled_emails:
    res = classify_email(em, "combined", rules, stats_model, settings)
    labeled_rows.append({
        "email_id": em.email_id,
        "subject": em.subject,
        "predicted_label": res.prediction,
        "predicted_label_num": 1 if res.prediction == "spam" else 0,
        "rule_score": res.rule_score,
        "stat_score": res.stat_score,
        "combined_score": res.total_score,
        "threshold": res.threshold,
        "matched_rules": "; ".join(f"{m.rule_id}:{m.pattern}({m.weight:+.1f})" for m in res.matched_rules),
        "top_keywords": "; ".join(f"{k.token}({k.weight:+.2f})" for k in res.top_keywords),
    })

out_labeled = PROJECT_ROOT / "outputs/labeled_new_emails.csv"
with out_labeled.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(labeled_rows[0].keys()))
    writer.writeheader()
    writer.writerows(labeled_rows)

n_spam_pred = sum(1 for r in labeled_rows if r["predicted_label"] == "spam")
n_ham_pred = len(labeled_rows) - n_spam_pred
print(f"Đã gán nhãn {len(labeled_rows)} email mới -> spam={n_spam_pred}, ham={n_ham_pred} (lưu tại outputs/labeled_new_emails.csv)")
print("\n--- 5 email mẫu sau khi gán nhãn tự động ---")
for r in labeled_rows[:5]:
    print(f"{r['email_id']} | [{r['predicted_label'].upper():4s}] S_total={r['combined_score']:+.2f} (rule={r['rule_score']:+.2f}, stat={r['stat_score']:+.2f}) | Subject: {r['subject'][:60]}")

Đã gán nhãn 30 email mới -> spam=14, ham=16 (lưu tại outputs/labeled_new_emails.csv)

--- 5 email mẫu sau khi gán nhãn tự động ---
U001 | [SPAM] S_total=+8.98 (rule=+5.00, stat=+3.98) | Subject: if you want real vicodin read this
U002 | [SPAM] S_total=+3.61 (rule=+0.00, stat=+3.61) | Subject: naturally busty helen . . fickle
U003 | [HAM ] S_total=-9.00 (rule=-5.00, stat=-4.00) | Subject: entex adjustments for 5 / 00
U004 | [SPAM] S_total=+7.00 (rule=+3.00, stat=+4.00) | Subject: d - link airplus g 802 . 11 g wireless router $ 87 . 00
U005 | [HAM ] S_total=-2.16 (rule=+0.00, stat=-2.16) | Subject: broom , bristles up , flew


## 5. Demo Tương Tác — Phân loại & Giải thích 1 Email Bất Kỳ

In [7]:
def predict_single_email(subject: str, body: str, mode: str = "combined") -> None:
    em = Email(email_id="DEMO_01", subject=subject, body=body)
    res = classify_email(em, mode, rules, stats_model, settings)
    print("=" * 70)
    print(f"Tiêu đề   : {subject}")
    print(f"Chế độ    : {res.mode}")
    print(f"Dự đoán   : {res.prediction.upper()} (S_total={res.total_score:+.4f}, Ngưỡng T={res.threshold:+.2f})")
    print(f"Thành phần: S_rule={res.rule_score:+.4f} | S_stat={res.stat_score:+.4f}")
    rules_txt = ", ".join(f"{m.pattern}({m.weight:+.1f})" for m in res.matched_rules) or "Không có"
    kw_txt = ", ".join(f"{k.token}({k.weight:+.2f})" for k in res.top_keywords) or "Không có"
    print(f"Quy tắc   : {rules_txt}")
    print(f"Top 5 từ  : {kw_txt}")

# Ví dụ 1: Spam rõ ràng
predict_single_email(
    "WINNER!!! Claim your prize today",
    "Congratulations winner! Click here for your free cash bonus and limited offer."
)

# Ví dụ 2: Ham công việc rõ ràng
predict_single_email(
    "Meeting agenda for Friday - Enron Houston",
    "Hi team, please review the attached file for the HPL meter schedule update."
)

# Ví dụ 3: Email pha trộn cả tín hiệu Spam và Ham
predict_single_email(
    "Free lunch after meeting agenda review",
    "Hi team, we have free coffee after the Enron project meeting in Houston."
)

Tiêu đề   : WINNER!!! Claim your prize today
Chế độ    : combined
Dự đoán   : SPAM (S_total=+7.8958, Ngưỡng T=-0.75)
Thành phần: S_rule=+5.0000 | S_stat=+2.8958
Quy tắc   : claim your prize(+4.0), limited offer(+2.5), click here(+2.5), winner(+3.0), free(+1.5), cash(+2.0)
Top 5 từ  : limited(+2.00), click(+1.48), bonus(+1.39), free(+1.27), claim(+1.10)
Tiêu đề   : Meeting agenda for Friday - Enron Houston
Chế độ    : combined
Dự đoán   : HAM (S_total=-9.0000, Ngưỡng T=-0.75)
Thành phần: S_rule=-5.0000 | S_stat=-4.0000
Quy tắc   : enron(-2.5), houston(-1.5), meeting agenda(-2.5), attached file(-2.0), meter(-2.0), hpl(-2.5)
Top 5 từ  : attached(-2.00), enron(-2.00), file(-2.00), hpl(-2.00), meter(-2.00)
Tiêu đề   : Free lunch after meeting agenda review
Chế độ    : combined
Dự đoán   : HAM (S_total=-6.1843, Ngưỡng T=-0.75)
Thành phần: S_rule=-5.0000 | S_stat=-1.1843
Quy tắc   : free(+1.5), enron(-2.5), houston(-1.5), meeting agenda(-2.5)
Top 5 từ  : enron(-2.00), houston(-1.95), team(-1.